# CodeBERT quality only (no repeated energy measurement)

Run Notebook 03 first. This notebook trains the frozen `microsoft/codebert-base`
embedding model **once** per dataset, purely to get predictive quality
(accuracy, balanced accuracy, ROC-AUC, F1) with grouped five-fold cross-validation.

It calls `experiment_utils.train_codebert_dataset` directly -- the same function
that the measurement pipeline in `Notebooks/05_codebert_training_measurements.ipynb`
calls internally, just **without** the energy-measurement wrapper and **without**
repeating it 10 times (validated) / 3 times (significant).

Why this notebook exists: the quality numbers (accuracy, ROC-AUC, ...) are
deterministic given a fixed random seed, so repeating training does not change
them -- repeating is only useful for averaging out *energy measurement* noise
(see `scripts/measure_codebert.py`, which runs the repeated version on the
LRZ testbed with real hardware energy counters). Running the repeated version
locally would cost 10x/3x the embedding time for the same quality result, so use
this notebook instead whenever you just need the accuracy numbers quickly on
your laptop.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run Notebook 03 first to create Data/Processed.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiment_utils import train_codebert_dataset

PROCESSED_DIR = ROOT / "Data" / "Processed"
MAX_INPUT_LENGTH = 512
BATCH_SIZE = 8
FORCE_CPU = True  # set False to use a local GPU if you have one

DATASET_FILES = {
    "validated": "prepared_training_dataset.parquet",
    "significant": "prepared_significant_training_dataset.parquet",
}

### Train once per dataset and collect quality metrics

This reuses the exact grouped five-fold evaluation logic as the measurement
notebook, just invoked a single time with no resource sampling around it.

In [2]:
performance_tables = {}
for dataset_name, file_name in DATASET_FILES.items():
    print(f"Training CodeBERT head once on {dataset_name}...")
    results_dir = ROOT / "Results" / dataset_name
    model_dir = PROCESSED_DIR / "models" / dataset_name
    performance = train_codebert_dataset(
        dataset_name=dataset_name,
        dataset_path=PROCESSED_DIR / file_name,
        results_dir=results_dir,
        model_dir=model_dir,
        max_length=MAX_INPUT_LENGTH,
        batch_size=BATCH_SIZE,
        force_cpu=FORCE_CPU,
    )
    performance_tables[dataset_name] = performance
    display(performance)

Training CodeBERT head once on validated...


d:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36151.53it/s]


validated: 21,410 pairs, 40,776 unique code samples; embedding time 11837.8s


,model,feature_config,accuracy_mean,accuracy_std,roc_auc_mean,roc_auc_std,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std
0,codebert_frozen_embeddings_logistic_regression,codebert_embeddings,0.570668,0.007443,0.61187,0.009798,0.570785,0.007485,0.569679,0.010009


Training CodeBERT head once on significant...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23483.95it/s]


significant: 3,000 pairs, 5,848 unique code samples; embedding time 1762.0s


,model,feature_config,accuracy_mean,accuracy_std,roc_auc_mean,roc_auc_std,balanced_accuracy_mean,balanced_accuracy_std,f1_mean,f1_std
0,codebert_frozen_embeddings_logistic_regression,codebert_embeddings,0.641333,0.030354,0.702407,0.040327,0.64239,0.029961,0.649879,0.033479


### Combined summary

In [3]:
combined = pd.concat(
    [table.assign(dataset=name) for name, table in performance_tables.items()],
    ignore_index=True,
)
combined = combined[[
    "dataset", "model", "feature_config",
    "accuracy_mean", "accuracy_std",
    "balanced_accuracy_mean", "balanced_accuracy_std",
    "roc_auc_mean", "roc_auc_std",
    "f1_mean", "f1_std",
]]
output_path = ROOT / "Results" / "codebert_quality_only_summary.csv"
combined.to_csv(output_path, index=False)
print(f"Saved combined quality summary to {output_path}")
display(combined)

print(
    "\nNote: these quality numbers are identical to whatever the repeated "
    "measurement notebook/script will report for the same dataset, since "
    "CodeBERT's frozen encoder and the Logistic Regression head are both "
    "deterministic given the fixed random seed. Only the energy numbers differ "
    "by repetition; the accuracy here does not need re-measuring on the testbed."
)

Saved combined quality summary to d:\Habiba\Uni\W26\Bachelor Thesis\energy-aware-code-recommendation\Results\codebert_quality_only_summary.csv


,dataset,model,feature_config,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std,f1_mean,f1_std
0,validated,codebert_frozen_embeddings_logistic_regression,codebert_embeddings,0.570668,0.007443,0.570785,0.007485,0.611870,0.009798,0.569679,0.010009
1,significant,codebert_frozen_embeddings_logistic_regression,codebert_embeddings,0.641333,0.030354,0.642390,0.029961,0.702407,0.040327,0.649879,0.033479



Note: these quality numbers are identical to whatever the repeated measurement notebook/script will report for the same dataset, since CodeBERT's frozen encoder and the Logistic Regression head are both deterministic given the fixed random seed. Only the energy numbers differ by repetition; the accuracy here does not need re-measuring on the testbed.
